# Portfolio Reconciliation

This notebook compares the source positions with the computed positions and displays every reconciliation error. A missing computed position is shown with a null computed quantity.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

import pandas as pd
from IPython.display import display

from src.core.repositories.position_repository import PositionRepository
from src.core.repositories.source_position_repository import SourcePositionRepository
from src.core.services.portfolio_reconciliation import PortfolioReconciliationService

In [ ]:
source_position_repository = SourcePositionRepository(
    project_root / 'data/store/source_positions/source_positions.csv'
)
position_repository = PositionRepository(
    project_root / 'data/store/computed_positions/positions.csv'
)

reconciliation = PortfolioReconciliationService(
    source_position_repository=source_position_repository,
    position_repository=position_repository,
)
status = reconciliation.reconcile()

print(f'Reconciliation status: {status.status}')
print(f'Conflicts found: {len(status.conflicts)}')

In [ ]:
conflict_table = pd.DataFrame([
    {
        'asset_id': conflict.source_position.asset_id,
        'asset_name': conflict.source_position.asset_name,
        'evaluation_date': conflict.source_position.evaluation_date,
        'source_quantity': conflict.source_position.quantity,
        'computed_quantity': (
            conflict.position.current_quantity
            if conflict.position is not None
            else None
        ),
        'error': (
            'missing computed position'
            if conflict.position is None
            else 'quantity mismatch'
        ),
    }
    for conflict in status.conflicts
], columns=[
    'asset_id',
    'asset_name',
    'evaluation_date',
    'source_quantity',
    'computed_quantity',
    'error',
]).sort_values(['asset_id', 'asset_name'], na_position='last').reset_index(drop=True)

if conflict_table.empty:
    print('No reconciliation errors.')
else:
    display(conflict_table.style.set_properties(**{'background-color': '#ffe6e6'}))